In [1]:
import os
import gzip
from rdkit import Chem
import pandas as pd

# Directory containing .sdf.gz files
sdf_dir = "/Users/kaveen/Documents/Research/Work/SILICO/Docking/dhfrdocking_results_new"
output_csv = "All_TSLPdocking_top_pose_by_CNN_VS_sorted.csv"

# Initialize a list to store data
data = []

# Process each .sdf.gz file
for sdf_file in os.listdir(sdf_dir):
    if sdf_file.endswith(".sdf.gz"):
        compound_name = sdf_file.split("_docked.sdf.gz")[0]  # Extract compound name

        # Open the .sdf.gz file and extract information
        with gzip.open(os.path.join(sdf_dir, sdf_file), "rb") as f:
            suppl = Chem.ForwardSDMolSupplier(f)
            pose_count = 0  # Initialize pose counter

            for mol in suppl:
                if mol:
                    pose_count += 1  # Increment pose counter

                    # Extract properties
                    cnn_score = mol.GetProp("CNNscore") if mol.HasProp("CNNscore") else None
                    cnn_affinity = mol.GetProp("CNNaffinity") if mol.HasProp("CNNaffinity") else None

                    # Calculate CNN_VS
                    try:
                        cnn_score = float(cnn_score)
                        cnn_affinity = float(cnn_affinity)
                        cnn_vs = cnn_score * cnn_affinity
                    except (ValueError, TypeError):
                        cnn_vs = None

                    # Append data only if CNN_VS is valid
                    if cnn_vs is not None:
                        data.append({
                            "Compound": compound_name,
                            "Pose": f"Pose_{pose_count}",  # Dynamically generate pose identifier
                            "CNNscore": cnn_score,
                            "CNNaffinity": cnn_affinity,
                            "CNN_VS": cnn_vs
                        })

# Create a DataFrame from the collected data
df = pd.DataFrame(data)

# Ensure the DataFrame is not empty
if not df.empty:
    # Group by compound and select the top pose by CNN_VS
    top_df = df.loc[df.groupby("Compound")["CNN_VS"].idxmax()]

    # Sort the DataFrame by CNN_VS in descending order
    top_df = top_df.sort_values(by="CNN_VS", ascending=False)

    # Save the top poses to a CSV file
    top_df.to_csv(output_csv, index=False)
    print(f"Top poses by CNN_VS for each compound saved to '{output_csv}'")
else:
    print("No valid poses found with CNN_VS scores.")


[23:52:44] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] ERROR: Could not sanitize molecule ending on line 119
[23:52:44] ERROR: Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] ERROR: Could not sanitize molecule ending on line 248
[23:52:44] ERROR: Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] ERROR: Could not sanitize molecule ending on line 377
[23:52:44] ERROR: Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] ERROR: Could not sanitize molecule ending on line 506
[23:52:44] ERROR: Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:52:44] ERROR: Could not sanitize molecule ending on

Top poses by CNN_VS for each compound saved to 'All_TSLPdocking_top_pose_by_CNN_VS_sorted.csv'
